# Vežba 3: Linearna regresija od nule

Hipoteza, funkcija cene i gradijentni spust, ručno, u NumPy-ju i u biblioteci scikit-learn, uz učenje zasnovano na modelu i na instancama

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba3.html](https://isatvss.surge.sh/vezba3.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Zadatak sa ispita

### Zadatak 1. Greška modela

Za tačke sa ispita izračunati vrednost hipoteze, odstupanje za svaku tačku i funkciju cene.

In [ ]:
import numpy as np
import pandas as pd

x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
y = np.array([2.5, 3, 3.5, 6, 4.5, 5, 5.5, 4, 6.5, 7])

theta0, theta1 = 2, 0.5
h = theta0 + theta1 * x
m = len(x)
J = np.sum((h - y) ** 2) / (2 * m)

print(pd.DataFrame({"x": x, "y": y, "h(x)": h, "h(x) - y": h - y}).to_string(index=False))
print(f"\nJ(theta) = {J:.2f}")

### Zadatak 2. Učenje zasnovano na modelu i na instancama

Predvideti izlaz za ulaz 4,5 na dva načina: pomoću prave sa ispita i kao prosek izlaza pet najbližih tačaka (kNN, k = 5).

Model daje 4,25. Kod kNN postoji zamka: peti i šesti najbliži sused, x = 2 i x = 7, jednako su udaljeni od 4,5, pa peti sused nije jednoznačan. Uz tačku x = 2 prosek je 4,40, a uz x = 7 je 4,90. Na ispitu se izabrano pravilo navodi uz rezultat, na primer bira se tačka sa manjim x, ili se uzimaju obe pa se računa prosek šest tačaka.

In [ ]:
x_novo = 4.5
print(f"Učenje zasnovano na modelu: h({x_novo}) = {theta0 + theta1 * x_novo:.2f}")

rastojanja = np.abs(x - x_novo)
redosled = np.argsort(rastojanja, kind="stable")
print("\nTačke po rastojanju od 4.5:")
print(pd.DataFrame({"x": x[redosled], "y": y[redosled], "rastojanje": rastojanja[redosled]}).head(7).to_string(index=False))

k = 5
najblizi = redosled[:k]
print(f"\nkNN, k = {k}, prosek y za x = {sorted(x[najblizi].tolist())}: {y[najblizi].mean():.2f}")
druga_mogucnost = np.append(najblizi[:4], np.where(x == 7)[0])
print(f"kNN, k = {k}, prosek y za x = {sorted(x[druga_mogucnost].tolist())}: {y[druga_mogucnost].mean():.2f}")

## Gradijentni spust u NumPy-ju

### Zadatak 3. Implementacija gradijentnog spusta

Napisati funkciju koja polazi od nultih parametara, ponavlja korak gradijentnog spusta i pamti vrednost funkcije cene u svakoj iteraciji. Primeniti je na tačke sa ispita i nacrtati kako se cena menja.

Gradijentni spust ne stiže do prave sa ispita, nego do prave \(h_\theta(x) = 2{,}533 + 0{,}403x\), čija je cena 0,3612, manja od 0,40. Prava sa ispita prolazi kroz osam tačaka, ali kvadratna greška kažnjava velika odstupanja, pa optimalna prava malo popušta svuda da bi smanjila odstupanja kod x = 4 i x = 8. Na ovom primeru se vidi i osetljivost metode najmanjih kvadrata na tačke koje odstupaju.

In [ ]:
import matplotlib.pyplot as plt


def gradijentni_spust(x, y, alfa, iteracija):
    t0, t1 = 0.0, 0.0
    m = len(x)
    istorija = []
    for _ in range(iteracija):
        greska = t0 + t1 * x - y
        istorija.append(np.sum(greska**2) / (2 * m))
        t0, t1 = t0 - alfa * np.sum(greska) / m, t1 - alfa * np.sum(greska * x) / m
    return t0, t1, istorija


t0, t1, istorija = gradijentni_spust(x, y, alfa=0.02, iteracija=3000)
for i in [0, 1, 10, 100, 1000, 2999]:
    print(f"iteracija {i:4d}: J = {istorija[i]:.4f}")
print(f"\nNaučeno: theta0 = {t0:.3f}, theta1 = {t1:.3f}")

plt.figure(figsize=(7, 3.5))
plt.plot(istorija, color="#5b3cc4")
plt.yscale("log")
plt.title("Funkcija cene tokom gradijentnog spusta")
plt.xlabel("Iteracija")
plt.ylabel("J(θ), logaritamska skala")
plt.show()

### Zadatak 4. Stopa učenja

Pokrenuti 60 iteracija gradijentnog spusta sa četiri stope učenja i uporediti tok funkcije cene.

In [ ]:
fig, (levo, desno) = plt.subplots(1, 2, figsize=(11, 3.8))
for alfa, boja in [(0.001, "#9aa5b1"), (0.01, "#5b3cc4"), (0.03, "#2b8a3e")]:
    _, _, ist = gradijentni_spust(x, y, alfa=alfa, iteracija=60)
    levo.plot(ist, label=f"α = {alfa}", color=boja)
    print(f"alfa = {alfa}: J posle 60 iteracija = {ist[-1]:.3g}")
levo.set_title("Stopa učenja dovoljno mala: J opada")
levo.set_xlabel("Iteracija")
levo.set_ylabel("J(θ)")
levo.legend()

_, _, ist = gradijentni_spust(x, y, alfa=0.06, iteracija=60)
print(f"alfa = 0.06: J posle 60 iteracija = {ist[-1]:.3g}")
desno.plot(ist, color="#d9480f", label="α = 0.06")
desno.set_yscale("log")
desno.set_title("Stopa učenja prevelika: J raste")
desno.set_xlabel("Iteracija")
desno.set_ylabel("J(θ), logaritamska skala")
desno.legend()

plt.tight_layout()
plt.show()

### Zadatak 5. Poređenje sa scikit-learn

Istrenirati `LinearRegression` na istim tačkama, uporediti parametre i nacrtati obe prave.

In [ ]:
from sklearn.linear_model import LinearRegression

regresija = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"scikit-learn:       theta0 = {regresija.intercept_:.3f}, theta1 = {regresija.coef_[0]:.3f}")
print(f"gradijentni spust:  theta0 = {t0:.3f}, theta1 = {t1:.3f}")

plt.figure(figsize=(6, 4))
plt.scatter(x, y, color="#d9480f", zorder=3, label="podaci")
plt.plot(x, theta0 + theta1 * x, color="#9aa5b1", linestyle="--", label="prava sa ispita")
plt.plot(x, t0 + t1 * x, color="#5b3cc4", label="naučena prava")
plt.xlabel("Ulazni parametar")
plt.ylabel("Izlazni parametar")
plt.legend(loc="lower right")
plt.show()

## Regresija na stvarnim podacima

### Zadatak 6. PM10 iz PM2.5 i uloga skaliranja

Na danima do kraja 2022. naučiti zavisnost koncentracije PM10 od PM2.5, jednom bez skaliranja, a jednom sa standardizovanim ulazom. Zatim modelom predvideti PM10 za 2023. i 2024. i izračunati grešku.

In [ ]:
vazduh = pd.read_csv("https://isatvss.surge.sh/podaci/vazduh_nis.csv", parse_dates=["datum"])
vazduh = vazduh.dropna(subset=["pm10", "pm25"])
trening = vazduh[vazduh["datum"].dt.year <= 2022]
test = vazduh[vazduh["datum"].dt.year >= 2023]

x_tr, y_tr = trening["pm25"].to_numpy(), trening["pm10"].to_numpy()
x_te, y_te = test["pm25"].to_numpy(), test["pm10"].to_numpy()

for alfa in [0.0001, 0.001]:
    a0, a1, ist = gradijentni_spust(x_tr, y_tr, alfa=alfa, iteracija=5000)
    print(f"bez skaliranja, alfa = {alfa}: theta0 = {a0:.2f}, theta1 = {a1:.3f}, J = {ist[-1]:.2f}")

sredina, odstupanje = x_tr.mean(), x_tr.std()
z_tr = (x_tr - sredina) / odstupanje
b0, b1, ist = gradijentni_spust(z_tr, y_tr, alfa=0.1, iteracija=200)
print(f"sa skaliranjem, alfa = 0.1, 200 iteracija: theta0 = {b0:.2f}, theta1 = {b1:.3f}, J = {ist[-1]:.2f}")
print(f"u originalnim jedinicama: PM10 = {b0 - b1 * sredina / odstupanje:.2f} + {b1 / odstupanje:.3f} * PM2.5")

h_te = b0 + b1 * (x_te - sredina) / odstupanje
print(f"\nJ na testu (2023-2024): {np.sum((h_te - y_te) ** 2) / (2 * len(y_te)):.2f}")
print(f"Prosečna apsolutna greška na testu: {np.mean(np.abs(h_te - y_te)):.2f} µg/m³")

### Zadatak 7. Više atributa i vektorski zapis

Predvideti PM10 iz četiri meteorološke veličine. Gradijentni spust napisati u vektorskom obliku, sa matricom atributa kojoj je dodata kolona jedinica, i uporediti parametre sa `LinearRegression`.

Meteorološki model greši u proseku 15,51 µg/m³, a osnovni model, koji uvek predviđa prosek treninga, 18,18 µg/m³. Vreme objašnjava deo promena zagađenja, ali mali. Na vežbi 4 model se proširuje i ocenjuje odgovarajućim metrikama.

In [ ]:
def gradijentni_spust_vektorski(X, y, alfa, iteracija):
    X1 = np.column_stack([np.ones(len(X)), X])
    theta = np.zeros(X1.shape[1])
    m = len(y)
    for _ in range(iteracija):
        theta = theta - alfa / m * X1.T @ (X1 @ theta - y)
    return theta


meteo = ["temperatura", "vetar_max", "padavine", "vlaznost"]
sredine, odstupanja = trening[meteo].mean(), trening[meteo].std(ddof=0)
X_tr = ((trening[meteo] - sredine) / odstupanja).to_numpy()

theta = gradijentni_spust_vektorski(X_tr, y_tr, alfa=0.1, iteracija=2000)
sk = LinearRegression().fit(X_tr, y_tr)

print(pd.DataFrame({"gradijentni_spust": theta, "scikit_learn": np.r_[sk.intercept_, sk.coef_]}, index=["theta0"] + meteo).round(3))

X_te = ((test[meteo] - sredine) / odstupanja).to_numpy()
h_meteo = np.column_stack([np.ones(len(X_te)), X_te]) @ theta
print(f"\nProsečna apsolutna greška na testu, meteorološki model: {np.mean(np.abs(h_meteo - y_te)):.2f} µg/m³")
print(f"Prosečna apsolutna greška na testu, prosek treninga:     {np.mean(np.abs(y_tr.mean() - y_te)):.2f} µg/m³")

## Zadaci za samostalan rad

### Zadatak 8. Jedan korak ručno

Za tačke (1, 2), (2, 3) i (3, 5), početne parametre jednake nuli i stopu učenja 0,1 ručno izračunati parametre posle jednog koraka gradijentnog spusta. Rezultat proveriti funkcijom iz zadatka 3.

### Zadatak 9. Koja prava je bolja

Izračunati funkciju cene na tačkama sa ispita za prave h(x) = 2 + 0,5x, h(x) = 1,5 + 0,6x i pravu dobijenu gradijentnim spustom. Koja prava ima najmanju cenu i da li je to prava koja vizuelno najbolje prati tačke?

### Zadatak 10. kNN za različito k

Predvideti izlaz za ulaz 4,5 metodom kNN za k = 1, 2, 3, 5 i 10. Šta postaje predviđanje kada je k jednako broju tačaka?

### Zadatak 11. Koliko iteracija bez skaliranja

Ponoviti učenje modela PM10 iz PM2.5 bez skaliranja, sa stopom učenja 0,001, za 5000, 50000 i 500000 iteracija, i uporediti parametre sa onima dobijenim posle skaliranja. Koliko puta više iteracija je potrebno bez skaliranja?

### Zadatak 12. Tumačenje parametara

Za model iz zadatka 7 ispisati parametre uz meteorološke veličine i standardne devijacije tih veličina. Šta znači znak parametra uz temperaturu i zašto se parametri modela sa standardizovanim atributima mogu međusobno porediti po veličini?